In [1]:
using HyQMOM, Plots, LinearAlgebra, LaTeXStrings

In [2]:
# create equations to get dc
N_vector = [10, 20, 30, 50, 100, 150, 200, 250, 1000] 
Kn = 1.0 # not relevant
c_l = -6.0
c_u = 6.0
domain = (-5.0, 5.0)
ρ_L = 7.0; v_L = 0.0; θ_L = 1.0 # not relevant
ρ_R = 1.0; v_R = 0.0; θ_R = 1.0 # not relevant
T_end = 0.3 # not relevant

0.3

In [3]:
function time_blocks(file)
    f = open(file)
    lines = readlines(f)
    n_vars = length(split(lines[3]))-1 # counts coordinates x as var

    # split into time-level blocks
    i = 1
    time = []
    while i < length(lines)
        l = lines[i]
        if startswith(l, "# timestep")# && endswith(l, "t=$T_end")
            j = findfirst("points=", l).stop
            n_points = parse(Int, l[j+1:j+findfirst(",", l[j:end]).start-2])
            push!(time, parse(Float64, split(l, "=")[end]))
            i += n_points
        else
            i += 1
        end
    end
    
    return time
end

time_blocks (generic function with 1 method)

In [4]:
function error(N, Kn, c_l, c_u, domain, ρ_L, v_L, θ_L, ρ_R, v_R, θ_R, file)
    blocks = readsol(file);
    basis, mesh, equations, initial_condition, solver, boundary_conditions = setupBGK1DRiemann(
        N, Kn,
        c_l, c_u, 
        Maxwellian(ρ_L, v_L, θ_L), # Density, velocity, temperature
        Maxwellian(ρ_R, v_R, θ_R);
        base_tree_level = 8,
        domain = domain,
    )

    L2_error_ρ = []; L2_error_v = []; L2_error_p = [];
    for time in 1:length(blocks)
        x = vec(blocks[time][:, 1]);
        conservative_moments = blocks[time][:, 2:end]'

        # macroscopic quantities
        ρ = HyQMOM.dc(equations) * sum(conservative_moments, dims=1);
        v = HyQMOM.dc(equations) * sum(conservative_moments .* equations.c_vec, dims=1) ./ ρ;
        θ = HyQMOM.dc(equations) * sum(conservative_moments .* (equations.c_vec .- v).^ 2, dims=1) ./ ρ;
        p = ρ .* θ;

        ρ = vec(ρ); v = vec(v); θ = vec(θ); p = vec(p);

        # Maxwellian distribution and reconstruction
        f_Maxwellian = [Maxwellian(ρ, v, θ).(equations.c_vec) for (ρ, v, θ) in zip(ρ, v, θ)];
        ρ_Maxwellian = [HyQMOM.dc(equations) * sum(f_Maxwellian[i,:][1]) for i in 1:size(f_Maxwellian, 1)]; # density
        v_Maxwellian = [HyQMOM.dc(equations) * sum(f_Maxwellian[i,:][1] .* equations.c_vec) / ρ_Maxwellian[i] for i in 1:size(f_Maxwellian, 1)]; # velocity
        θ_Maxwellian = [HyQMOM.dc(equations) * sum(f_Maxwellian[i,:][1] .* (equations.c_vec .- v_Maxwellian[i]).^ 2) / ρ_Maxwellian[i] for i in 1:size(f_Maxwellian, 1)];
        p_Maxwellian = ρ_Maxwellian .* θ_Maxwellian;

        # L2 errors
        push!(L2_error_ρ, norm(ρ .- ρ_Maxwellian, 2) / norm(ρ_Maxwellian, 2))
        push!(L2_error_v, norm(v .- v_Maxwellian, 2) / norm(v_Maxwellian, 2))
        push!(L2_error_p, norm(p .- p_Maxwellian, 2) / norm(p_Maxwellian, 2))
    end

    return L2_error_ρ, L2_error_v, L2_error_p
end

error (generic function with 1 method)

In [5]:
L2_ρ, L2_v, L2_p = [], [], []
for N in N_vector
    file = "../out/Riemann1D/bgk_solution_N$(N)_Kn$(Kn)_T_end$(T_end)_rho_L$(ρ_L)_rho_R$(ρ_R)_v_L$(v_L)_v_R$(v_R)_theta_L$(θ_L)_theta_R$(θ_R).tsv"
    L2_error_ρ, L2_error_v, L2_error_p = error(N, Kn, c_l, c_u, domain, ρ_L, v_L, θ_L, ρ_R, v_R, θ_R, file)

    timeblock = time_blocks(file)
    plt = plot(
        timeblock, L2_error_ρ;
        xlabel = "t",
        label = "ρ",
        color = :blue,
        legend = :topleft,
    )
    plot!(
        plt, timeblock, L2_error_v;
        label = "v",
        color = :red,
    )
    plot!(
        plt, timeblock, L2_error_p;
        label = "p",
        color = :green,
    )
    plot!(plt, yaxis=:log)
    title!(plt, "L2 Relative Errors for N=$(N)")
    xlabel!(plt, "Time t")
    ylabel!(plt, "L2 Relative Error")
    display(plt)
    savefig(plt, "../out/Riemann1D/bgk_L2_Relative_Errors_N$(N)_Kn$(Kn)_T_end$(T_end)_rho_L$(ρ_L)_rho_R$(ρ_R)_v_L$(v_L)_v_R$(v_R)_theta_L$(θ_L)_theta_R$(θ_R).pdf")

    push!(L2_ρ, L2_error_ρ[end])
    push!(L2_v, L2_error_v[end])
    push!(L2_p, L2_error_p[end])
end

plt = plot(
    N_vector, L2_ρ;
    xlabel = "N",
    label = "ρ",
    color = :blue,
    marker = :circle,
    legend = :topright,
)
plot!(
    plt, N_vector, L2_v;
    label = "v",
    color = :red,
    marker = :circle,
)
plot!(
    plt, N_vector, L2_p;
    label = "p",
    color = :green,
    marker = :circle,
)
plot!(plt, yaxis=:log)
# title!(plt, "L2 Relative Errors at Final Time T=$(T_end)")
xlabel!(plt, L"N_c")
ylabel!(plt, L"L_2")
display(plt)
savefig(plt, "../out/Riemann1D/bgk_L2_Relative_Errors_T_end$(T_end)_rho_L$(ρ_L)_rho_R$(ρ_R)_v_L$(v_L)_v_R$(v_R)_theta_L$(θ_L)_theta_R$(θ_R).pdf")

SystemError: SystemError: opening file "../out/Riemann1D/bgk_solution_N10_Kn1.0_T_end0.3_rho_L7.0_rho_R1.0_v_L0.0_v_R0.0_theta_L1.0_theta_R1.0.tsv": No such file or directory